In [1]:
# Install libraries 
!pip install seaborn --quiet
!pip install missingno --quiet
!pip install imblearn --quiet
!pip install scikit-learn --quiet

In [2]:
# Import required libraries 
import os 
import sys
import pandas as pd 
import seaborn as sns
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
import warnings

sys.path.append(os.path.abspath(".."))

# Import functions 
import functions.wrangling as wrg
import functions.missing_labs as ml
import functions.eda as eda
import functions.eda_model as em

warnings.filterwarnings("ignore")

# Set working directory (change this to the folder on your system)
os.chdir(r"G:\.shortcut-targets-by-id\1qO0AfYMqzVbXreDMm-gZUvrYXtVZCnDA\CHL8010F2  CPCSSN Dataset") 

In [4]:
# Load and clean datasets

# Define and load file paths 
file_paths = {
    'patient': 'C4MPatient.csv',
    'lab': 'C4MLab.csv',
    'diag': 'C4MEncounterdiagnosis.csv',
    'condition': 'C4MHealthCondition.csv'
}
datasets = wrg.load_csv(file_paths)

# Specify columns to keep from each dataset
columns_to_keep = {
    'patient': ["Patient_ID", "Sex", "BirthYear"],
    'lab': ["Patient_ID", "Name_calc", "TestResult_calc", "PerformedDate"],
    'diag': ["Patient_ID", "DiagnosisText_calc", "DiagnosisCode_calc", "DateCreated"],
    'condition': ["Patient_ID", "DiagnosisText_calc", "DateCreated"]
}
datasets = wrg.select_columns(datasets, columns_to_keep)

# Clean diagnosis data 
datasets['diag'] = wrg.replace_string_nan(datasets['diag'], 'DiagnosisCode_calc')
datasets['diag'] = wrg.preprocess_data(datasets['diag'], [
    ('DiagnosisText_calc', 'uppercase'),
    ('DiagnosisCode_calc', 'strip'),
    ('DiagnosisCode_calc', 'dropna'),
    ('DateCreated', 'datetime')
])

In [9]:
def filter_labs_within_window(df, date_col, ref_date_col, window_days):
    """Filter rows where date_col is within a fixed window after ref_date_col
    
    Parameters:
    - df: Dataframe with time-based events (like labs)
    - date_col: column name for event date 
    - ref_date_col: column name for reference date (e.g., diagnosis date)
    - window_days: number of days after ref_date_col to include 

    Returns:
    - filtered dataframe with events within the window 
    """
    return df[
        (df[date_col] >= df[ref_date_col]) &
        (df[date_col] <= df[ref_date_col] + pd.Timedelta(days=window_days))
    ]

def get_all_labs_from_first_date(df, id_col, date_col):
    """
    Get all lab rows from each patient's first available lab date

    Parameters:
    - df: DataFrame of labs
    - id_col: patient ID column 
    - date_col: column name for lab test date

    Returns:
    - DataFrame with all lab rows from the earliest lab date per patient
    """
    first_dates = df.groupby(id_col)[date_col].min().reset_index()
    return df.merge(first_dates, on=[id_col, date_col], how='inner')


In [11]:
# Extract bipolar disorder lab results and handle missing cases

# Define BD ICD-9 codes and relevant lab markers
bd_codes = ["296.0", "296.1", "296.4", "296.5", "296.6", "296.7", "296.80", "296.89"]
relevant_markers = ["TOTAL CHOLESTEROL", "HBA1C", "HDL", "FASTING GLUCOSE", "LDL", "INR", "GLUCOSE TOLERANCE"]

# Extract lab results that occur after BD diagnosis (filtered by relevant markers)
bd_labs_after = wrg.extract_labs_relative_to_diagnosis(
    lab_df=datasets['lab'],
    diag_df=datasets['diag'],
    diagnosis_codes=bd_codes,
    lab_test_names=relevant_markers
)

# Get the first BD diagnosis date per patient
first_dx = wrg.get_first_matching_diagnosis(
    df=datasets['diag'],
    diagnosis_col='DiagnosisCode_calc',
    date_col='DateCreated',
    target_codes=bd_codes,
    new_date_col='BD_Diagnosis_Date',
    new_code_col='BD_Code'
)

# Merge diagnosis info into lab results
bd_labs_after = bd_labs_after.merge(
    first_dx[['Patient_ID', 'BD_Diagnosis_Date', 'BD_Code']],
    on='Patient_ID', how='left'
).drop(columns=['Lab_Timing'])

# Filter labs that occurred within 1 year of BD diagnosis
bd_year_labs_after = filter_labs_within_window(
    df=bd_labs_after,
    date_col='PerformedDate',
    ref_date_col='BD_Diagnosis_Date',
    window_days=365
)

# If patients don't have a lab result within the 1 year of BD diagnosis, 
# get labs from the first available lab date after diagnosis
patients_within_1yr = set(bd_year_labs_after['Patient_ID'])
patients_without_1yr_labs = bd_labs_after[~bd_labs_after['Patient_ID'].isin(patients_within_1yr)]
first_available_labs_after_1yr = get_all_labs_from_first_date(
    df=patients_without_1yr_labs,
    id_col='Patient_ID',
    date_col='PerformedDate'
)

bd_combined = pd.concat([bd_year_labs_after, first_available_labs_after_1yr], ignore_index=True)

# Remove duplicate entries of same test on same day, keeping entries with more lab results
bd_combined = (
    bd_combined.sort_values(by=['Patient_ID', 'PerformedDate', 'Name_calc', 'TestResult_calc'], na_position='last')
    .drop_duplicates(subset=['Patient_ID', 'PerformedDate', 'Name_calc'], keep='first')
)

# Check for repeated tests on same day, such as multiple entries for the same test
dup_tests_same_day = (
    bd_combined.groupby(['Patient_ID', 'PerformedDate', 'Name_calc'])
    .size()
    .reset_index(name='n')
)
dup_tests_same_day = dup_tests_same_day[dup_tests_same_day['n'] > 1]
patients_with_dup_tests = dup_tests_same_day['Patient_ID'].nunique()

# Pivot to wide format
bd_labs_within_1yr_or_first_avail = wrg.pivot_lab_data(
    df=bd_combined,
    index_cols=['Patient_ID', 'PerformedDate', 'BD_Diagnosis_Date', 'BD_Code'],
    name_col='Name_calc',
    value_col='TestResult_calc'
).sort_values(['Patient_ID', 'PerformedDate'])

# Add age and sex data for each patient
bd_labs_within_1yr_or_first_avail = wrg.add_demographics_to_labs(
    lab_df=bd_labs_within_1yr_or_first_avail,
    patient_df=datasets['patient']
)

# How many patients didn't have a lab test within 1 year of diagnosis
num_no_labs_1yr = first_available_labs_after_1yr['Patient_ID'].nunique()

summary_stats = [
    ("Total BD patients (first-ever diagnosis)", first_dx['Patient_ID'].nunique()),
    ("Patients with labs AFTER BD diagnosis", bd_labs_after['Patient_ID'].nunique()),
    ("Patients with NO labs within 1 year (included using first available)", num_no_labs_1yr),
    ("Patients with MULTIPLE entries for SAME test on SAME day", patients_with_dup_tests)
]
wrg.print_summary_stats(summary_stats)
bd_labs_within_1yr_or_first_avail.to_csv("bd_labs_1yr_or_first_avail.csv",index=False)
#bd_labs_within_1yr_or_first_avail

Total BD patients (first-ever diagnosis): 378
Patients with labs AFTER BD diagnosis: 214
Patients with NO labs within 1 year (included using first available): 98
Patients with MULTIPLE entries for SAME test on SAME day: 0


In [12]:
# LABELLING PATIENTS WITH COMORBIDITIES (non-BD diagnosis on same day as BD diagnosis) and ONLY 
# looking at their first lab result (keep multiple labs if they occurred on the same, first lab day)

def get_all_labs_from_first_date(df, id_col, date_col):
    """
    Get all lab rows from each patient's first available lab date

    Parameters:
    - df: DataFrame of lab results
    - id_col: column name for patient ID
    - date_col: column name for lab test date

    Returns:
    - DataFrame containing all lab rows from each patient’s earliest test date
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    first_dates = df.groupby(id_col)[date_col].min().reset_index()
    return df.merge(first_dates, on=[id_col, date_col], how='inner')

def create_lab_summary_comorbidity(
    lab_pivot_df,
    diagnosis_df,
    first_dx_df,
    target_codes,
    lab_marker_cols,
    comorbidity_timing="same_day",  # or "after", "before"
    code_col='DiagnosisCode_calc',
    date_col='DateCreated',
    patient_col='Patient_ID',
    diag_date_col='BD_Diagnosis_Date',
    demographic_cols=['Age', 'Sex']
):
    """
    Add comorbidity label and return summary lab table.

    Only patients in lab_pivot_df will be considered for comorbidity.

    Returns:
    - summary_df: final lab table with comorbidity flag
    - num_comorbid: count of patients with comorbidities in lab cohort
    """
    lab_patients = set(lab_pivot_df[patient_col])

    diag = diagnosis_df.copy()
    diag[date_col] = pd.to_datetime(diag[date_col], errors='coerce')
    diag = diag.merge(first_dx_df[[patient_col, diag_date_col]], on=patient_col, how='left')

    if comorbidity_timing == "same_day":
        condition = diag[date_col] == diag[diag_date_col]
    elif comorbidity_timing == "after":
        condition = diag[date_col] > diag[diag_date_col]
    elif comorbidity_timing == "before":
        condition = diag[date_col] < diag[diag_date_col]
    else:
        raise ValueError("Invalid comorbidity_timing. Choose 'same_day', 'after', or 'before'.")

    comorbid_patients = diag[
        (diag[patient_col].isin(lab_patients)) &  # <--- This is the critical line
        condition &
        (~diag[code_col].isin(target_codes))
    ][patient_col].unique()

    lab_pivot_df = lab_pivot_df.copy()
    lab_pivot_df['Comorbidity'] = lab_pivot_df[patient_col].isin(comorbid_patients).astype(int)

    summary_cols = ['Patient_ID', 'PerformedDate', 'BD_Diagnosis_Date', 'BD_Code', 'Comorbidity'] + lab_marker_cols + demographic_cols
    summary_df = lab_pivot_df[summary_cols]

    return summary_df, len(comorbid_patients)

In [15]:
# Define BD ICD-9 codes and relevant lab markers
bd_codes = ["296.0", "296.1", "296.4", "296.5", "296.6", "296.7", "296.80", "296.89"]
relevant_markers = ["TOTAL CHOLESTEROL", "HBA1C", "HDL", "FASTING GLUCOSE", "LDL", "INR", "GLUCOSE TOLERANCE"]

# Extract lab results that occur after BD diagnosis
bd_labs_after = wrg.extract_labs_relative_to_diagnosis(
    lab_df=datasets['lab'],
    diag_df=datasets['diag'],
    diagnosis_codes=bd_codes,
    lab_test_names=relevant_markers
)

# Get first BD diagnosis per patient
first_dx = wrg.get_first_matching_diagnosis(
    df=datasets['diag'],
    diagnosis_col='DiagnosisCode_calc',
    date_col='DateCreated',
    target_codes=bd_codes,
    new_date_col='BD_Diagnosis_Date',
    new_code_col='BD_Code'
)

# Merge diagnosis info into lab results
bd_labs_after = bd_labs_after.merge(
    first_dx[['Patient_ID', 'BD_Diagnosis_Date', 'BD_Code']],
    on='Patient_ID', how='left'
).drop(columns=['Lab_Timing'])

# Get all labs from the patient's first lab date after BD diagnosis
first_lab_df = get_all_labs_from_first_date(
    df=bd_labs_after,
    id_col='Patient_ID',
    date_col='PerformedDate'
)

# Remove duplicate entries of same test on same day (keep non-NaN)
first_lab_df = (
    first_lab_df.sort_values(by=['Patient_ID', 'PerformedDate', 'Name_calc', 'TestResult_calc'], na_position='last')
    .drop_duplicates(subset=['Patient_ID', 'PerformedDate', 'Name_calc'], keep='first')
)

# Pivot to wide format
first_lab_pivot = wrg.pivot_lab_data(
    df=first_lab_df,
    index_cols=['Patient_ID', 'PerformedDate', 'BD_Diagnosis_Date', 'BD_Code'],
    name_col='Name_calc',
    value_col='TestResult_calc'
).sort_values(['Patient_ID', 'PerformedDate'])

# Add age and sex
first_lab_pivot = wrg.add_demographics_to_labs(
    lab_df=first_lab_pivot,
    patient_df=datasets['patient']
)

# Create summary table with comorbidity flag (same-day non-BD diagnosis)
bd_lab_after_comorbidity, num_comorbid = create_lab_summary_comorbidity(
    lab_pivot_df=first_lab_pivot,
    diagnosis_df=datasets['diag'],
    first_dx_df=first_dx,
    target_codes=bd_codes,
    lab_marker_cols=relevant_markers,
    comorbidity_timing="same_day"
)

summary_stats = [
    ("Patients with labs AFTER BD diagnosis", bd_labs_after['Patient_ID'].nunique()),
    ("Patients with NON-BD diagnosis on SAME DAY as BD (comorbidity)", num_comorbid)
]
wrg.print_summary_stats(summary_stats)
bd_lab_after_comorbidity.to_csv("bd_lab_after_comorbidity.csv",index=False)
#bd_lab_after_comorbidity

Patients with labs AFTER BD diagnosis: 214
Patients with NON-BD diagnosis on SAME DAY as BD (comorbidity): 71
